<a href="https://colab.research.google.com/github/mufaddalkc/genai-foundations-helpdesk-assistant/blob/main/Skill_Builder_Lab_GenAI_Foundations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment: Build an AI Student Helpdesk Assistant


> **API Compatibility Note:** This notebook is configured to work out-of-the-box with a standard `DEEPSEEK_API_KEY` (using `https://api.deepseek.com/chat/completions` and `deepseek-chat`). For development and testing, the setup cell in Step 0 also automatically detects Groq API keys (`gsk_...`) stored under the same `DEEPSEEK_API_KEY` secret name and routes requests to the Groq endpoint accordingly. No code changes are required for grading.


## Business Scenario
Northstar Learning Hub has a handbook containing its student policies. The academic team wants a basic AI assistant that can answer learners' questions **using the handbook**, instead of guessing.

You will build a small pipeline that:
1. Reads the provided PDF and divides it into text chunks.
2. Tests **zero-shot and few-shot prompts** to categorize student questions.
3. Calls the **DeepSeek API** from Python.
4. Retrieves a relevant handbook chunk using **keyword overlap**.
5. Uses a **LangChain chain** to generate a grounded answer.
6. Tests whether answers are supported by the PDF, including for missing information.


## Learning Outcomes


By completing this assignment, you should be able to:
- Apply system/user messages and zero-shot/few-shot prompting.
- Send API requests using `requests` with configurable `temperature` and `max_tokens`.
- Extract and chunk PDF text with Python and `pypdf`.
- Retrieve a relevant chunk with simple word-overlap scoring.
- Connect `PromptTemplate`, `RunnableLambda`, and `StrOutputParser`.
- Evaluate grounded answers and identify unsupported responses.



## Provided resources




- [Northstar_Student_Handbook.pdf](https://drive.google.com/file/d/13GKO3S8Z0ek6nOHqZeizAbqu3X9IDcsi/view?usp=drive_link)

- Example test questions near the end


## Submission



Upload your completed `.ipynb` with executed outputs on GitHub and share the link of repository.

## Part A — Prepare Your Colab Environment
### Step 0. Install libraries and load your API key

Run the setup cells below. The installer and key-loading code are provided because the assessed work begins with the functions in Step 1.

1. In Colab, open the **Secrets** sidebar.
2. Create a secret named `DEEPSEEK_API_KEY` and allow this notebook to use it.
3. Never paste your secret into a code cell or output.
4. Use a DeepSeek model available to your account. The default shown below is `deepseek-chat`; update it only if your account requires a different supported model.

**Purpose of Code:** This setup cell installs the required libraries, imports the tools used throughout the assignment, and loads your DeepSeek API key securely. Later functions depend on these imports and settings to run.


In [26]:
%pip -q install requests pypdf langchain-core

import os
import re
import requests
from typing import Any
from pypdf import PdfReader
from google.colab import files, userdata
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser

DEEPSEEK_API_KEY = userdata.get("DEEPSEEK_API_KEY")
if not DEEPSEEK_API_KEY:
    raise ValueError("Save DEEPSEEK_API_KEY in Colab Secrets and enable access.")

if DEEPSEEK_API_KEY.startswith("gsk_"):
    DEEPSEEK_API_URL = "https://api.groq.com/openai/v1/chat/completions"
    DEEPSEEK_MODEL = "openai/gpt-oss-120b"
    provider = "Groq"
else:
    DEEPSEEK_API_URL = "https://api.deepseek.com/chat/completions"
    DEEPSEEK_MODEL = "deepseek-chat"
    provider = "DeepSeek"

HEADERS = {
    "Authorization": f"Bearer {DEEPSEEK_API_KEY}",
    "Content-Type": "application/json",
}

print(f"Setup complete. API key loaded securely for {provider} (model: {DEEPSEEK_MODEL}).")

Setup complete. API key loaded securely for Groq (model: openai/gpt-oss-120b).


In [27]:
import requests

test_payload = {
    "model": DEEPSEEK_MODEL,
    "messages": [{"role": "user", "content": "Ping"}],
    "max_tokens": 5
}

try:
    # Updating headers using the loaded variables
    HEADERS = {
        "Authorization": f"Bearer {DEEPSEEK_API_KEY}",
        "Content-Type": "application/json",
    }
    response = requests.post(DEEPSEEK_API_URL, headers=HEADERS, json=test_payload, timeout=10)
    if response.status_code == 200:
        print("Success! Your Groq API key is valid and connected.")
        print("Response:", response.json()["choices"][0]["message"]["content"])
    else:
        print(f"API Key check failed with status code {response.status_code}.")
        print("Details:", response.text)
except Exception as e:
    print(f"An error occurred during verification: {e}")

Success! Your Groq API key is valid and connected.
Response: 


### Step 1. Upload the student handbook PDF

Download the **provided** handbook PDF and upload it in the next cell. Run this once each time the Colab runtime is reset. We use the file's original name in all later steps.

**Purpose of Code:** This cell makes the handbook PDF available inside the Colab runtime and saves its file path. The PDF-reading function in the next step will use this path to open the correct document.


In [28]:
uploaded = files.upload()
if not uploaded:
    raise ValueError("Please upload the supplied student handbook PDF.")

PDF_PATH = next(iter(uploaded))
if not PDF_PATH.lower().endswith(".pdf"):
    raise ValueError("The uploaded file must be a PDF.")
print("Uploaded:", PDF_PATH)

Saving Northstar_Student_Handbook.pdf to Northstar_Student_Handbook (1).pdf
Uploaded: Northstar_Student_Handbook (1).pdf


### Step 2. Extract text from the PDF

**Your task:** Complete `read_handbook_pdf(pdf_path)`.

- Open the file with `PdfReader`.
- Extract the text from every page.
- Ignore any empty page text.
- Join the extracted text into **one string**.
- Return the combined string.

**Expected input:** Path to one text-based PDF.  
**Expected output:** A non-empty string containing handbook text.

**Hint:** Recall `page.extract_text()` from the LangChain & RAG lesson. Scanned/image-only PDFs are out of scope.

**Purpose of Code:** This function converts the uploaded handbook from a PDF into plain text. The assistant needs this text so it can search the handbook and use its policies when answering student questions.


In [30]:
def read_handbook_pdf(pdf_path: str) -> str:
    """Return readable text extracted from all pages of the handbook PDF."""
    # TODO 1: Open the PDF using PdfReader.
    reader = PdfReader(pdf_path)
    # TODO 2: Extract and collect text from each page.
    pages_text = []
    for page in reader.pages:
        text = page.extract_text()
        if text and text.strip():
            pages_text.append(text)
    # TODO 3: Join the collected text into a single string.
    return "\n".join(pages_text)
    # TODO 3: Combine the page text and return it.

### Step 3. Split the document into chunks

**Your task:** Complete `split_text_into_chunks(text, chunk_size, chunk_overlap)`.

- Split the text into smaller, overlapping character-based chunks.
- Start from character index `0` and move by `chunk_size - chunk_overlap`.
- Remove empty chunks and return a list of strings.
- Check that overlap is smaller than chunk size.

**Expected output:** A list such as `['first chunk ...', 'second chunk ...']`.

**Hint:** The RAG lesson used a simple `start`, `end`, and `while` loop. Do not use a new text-splitting library.

**Purpose of Code:** This function breaks the long handbook text into smaller, overlapping sections. Smaller chunks make it easier to find relevant policy information without sending the entire handbook to the LLM.


In [31]:
def split_text_into_chunks(
    text: str,
    chunk_size: int = 1000,
    chunk_overlap: int = 150,
) -> list[str]:
    """Return overlapping character-based chunks from the PDF text."""
    # TODO 1: Validate that 0 <= chunk_overlap < chunk_size.
    if not (0 <= chunk_overlap < chunk_size):
        raise ValueError("chunk_overlap must be in [0, chunk_size).")

    # TODO 2: Slice the text using a moving start and end position.
    chunks = []
    start = 0
    step = chunk_size - chunk_overlap

    while start < len(text):
      end = start + chunk_size
      chunk = text[start:end].strip()

    # TODO 3: Collect non-empty chunks, then return them.
      if chunk:
        chunks.append(chunk)

      start += step

    return chunks

## Part B — Apply Prompt Engineering
### Step 4. Create a zero-shot question-classification prompt

**Your task:** Complete `build_zero_shot_prompt(question)`.

Write a prompt that asks the LLM to classify a question using **only** these categories:
`Enrollment`, `Assignments`, `Certificates`, `Support`, `Other`.

Requirements:
- Give a clear instruction but **no examples**.
- Include the student's question.
- Ask for **only the category name** in the response.

**Example input:** `"Can I resubmit my graded homework?"`  
**Expected category:** `Assignments`.

**Purpose of Code:** This function creates a zero-shot prompt that tells the LLM how to categorize a student question without examples. It provides a baseline for comparing different prompt-engineering techniques.


In [32]:
def build_zero_shot_prompt(question: str) -> str:
    """Return a zero-shot classification prompt for a student question."""
    # TODO 1: Explain the classification task and allowed labels.
    # TODO 2: Insert the student's question.
    # TODO 3: Specify the exact output format.
    return (
        "Classify the following student question into exactly one of these categories: "
        "Enrollment, Assignments, Certificates, Support, Other.\n\n"
        f"Question: {question}\n\n"
        "Output only the category name."
    )

### Step 5. Create a few-shot question-classification prompt

**Your task:** Complete `build_few_shot_prompt(question)`.

Use the **same categories** and output format as Step 4, but add at least **three labeled examples** in the prompt.

You may use these examples:

| Example question | Category |
|---|---|
| How do I enroll in the next batch? | Enrollment |
| When is my assignment due? | Assignments |
| I cannot open my learning portal. | Support |

End with the new question and ask for its category. Return the complete prompt as a string.

**Purpose of Code:** This function creates a few-shot prompt with labeled examples of student questions. You will compare it with the zero-shot version to observe how examples can influence classification.


In [33]:
def build_few_shot_prompt(question: str) -> str:
    """Return a few-shot classification prompt with labeled examples."""
    # TODO 1: State the allowed categories.
    # TODO 2: Include three or more question/category examples.
    # TODO 3: Add the new question and request only the category.
    return (
        "Classify the following student question into exactly one of these categories: "
        "Enrollment, Assignments, Certificates, Support, Other.\n\n"
        "Examples:\n"
        "Question: How do I enroll in the next batch?\n"
        "Category: Enrollment\n\n"
        "Question: When is my assignment due?\n"
        "Category: Assignments\n\n"
        "Question: I cannot open my learning portal.\n"
        "Category: Support\n\n"
        f"Question: {question}\n"
        "Category:\n\n"
        "Output only the category name."
    )

## Part C — Call an LLM Using an API
### Step 6. Implement a reusable DeepSeek API function

**Your task:** Complete `call_deepseek(messages, temperature, max_tokens)` using **Python `requests`**, as in the LLM API lesson.

- Prepare the JSON request body with `model`, `messages`, `temperature`, and `max_tokens`.
- Send the request with `requests.post` to `DEEPSEEK_API_URL` using `HEADERS` and `timeout=60`.
- Call `raise_for_status()`.
- Parse the response JSON and return the assistant's message content as a string.
- Include any model-specific settings only if needed by your configured DeepSeek model.

**Expected input:** A list of `system` and `user` messages.  
**Expected output:** Generated text.

**Security:** Keep the API key in Colab Secrets; do not print it.

**Purpose of Code:** This function sends messages to the DeepSeek model and retrieves its response. Making it reusable allows both the classification tasks and the RAG pipeline to call the same API without duplicating request code.


In [34]:
def call_deepseek(
    messages: list[dict],
    temperature: float = 0.2,
    max_tokens: int = 250,
) -> str:
    """Send messages to DeepSeek and return generated assistant text."""
    # TODO 1: Build the JSON payload.
    # TODO 2: Send an HTTP POST with requests.
    # TODO 3: Check status, decode JSON, return the response text.
    payload = {
        "model": DEEPSEEK_MODEL,
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens,
    }

    response = requests.post(DEEPSEEK_API_URL, headers=HEADERS, json=payload, timeout=60)
    response.raise_for_status()
    data = response.json()
    return data["choices"][0]["message"]["content"].strip()

### Step 7. Compare zero-shot and few-shot results

**Your task:** Complete `compare_classification_prompts(question)`.

- Call your zero-shot prompt function.
- Call your few-shot prompt function.
- Send each prompt to your `call_deepseek` function using the same low temperature.
- Return a dictionary containing `question`, `zero_shot`, and `few_shot`.

**Discussion:** Were the two results identical? If not, why might examples change the result?

**Note:** This step sends two API requests and may incur a small cost.

**Purpose of Code:** This function runs the same student question through zero-shot and few-shot prompts. Its output helps you compare their classifications and understand how prompt design affects LLM behavior.


In [35]:
def compare_classification_prompts(question: str) -> dict:
    """Return outputs from zero-shot and few-shot classification."""
    # TODO 1: Build both prompts.
    zero_prompt = build_zero_shot_prompt(question)
    few_prompt = build_few_shot_prompt(question)

    # TODO 2: Call DeepSeek separately for each prompt.
    zero_res = call_deepseek([{"role": "user", "content": zero_prompt}], temperature=0.0)
    few_res = call_deepseek([{"role": "user", "content": few_prompt}], temperature=0.0)

    # TODO 3: Return {'question': ..., 'zero_shot': ..., 'few_shot': ...}.
    return {
        "question": question,
        "zero_shot": zero_res,
        "few_shot": few_res,
    }

## Part D — Retrieve Relevant Context
### Step 8. Find the most relevant PDF chunk

**Your task:** Complete `retrieve_best_chunk(question, chunks)`.

Use **basic keyword overlap**, as demonstrated in the RAG lesson:
1. Convert the question to lowercase words using `re.findall(r"[a-zA-Z]+", ...)`.
2. Convert each chunk to lowercase words using the same method.
3. Score each chunk by the number of **shared unique words**.
4. Return the best chunk and its integer score as `(context, score)`.
5. If there are no chunks or every chunk has score `0`, return `("", 0)`.

**Do not use:** embeddings, vector databases, semantic-search packages, or the model to retrieve text.

**Why return a score?** It lets the caller notice obvious retrieval failures. A positive overlap score does **not** guarantee that the chunk contains the answer.

**Purpose of Code:** This function searches the handbook chunks for words that overlap with the student’s question. It selects useful context for the LLM without requiring embeddings or a vector database, and reports when no words match.


In [36]:
def retrieve_best_chunk(question: str, chunks: list[str]) -> tuple[str, int]:
    """Return the highest-overlap chunk and its keyword-overlap score."""
    # TODO 1: Convert the question to a set of lowercase words.
    if not chunks:
        return ("", 0)
    # TODO 2: Compare it with words from every chunk.
    q_words = set(re.findall(r"[a-zA-Z]+", question.lower()))
    best_chunk = ""
    best_score = 0

    for chunk in chunks:
        c_words = set(re.findall(r"[a-zA-Z]+", chunk.lower()))
        score = len(q_words.intersection(c_words))
        if score > best_score:
            best_chunk = chunk
            best_score = score
    # TODO 3: Return the chunk with highest score, or ('', 0) if none match.
    if best_score == 0:
      return ("", 0)
    return (best_chunk, best_score)

## Part E — Create the RAG Chain Using LangChain
### Step 9. Build the grounded prompt template

**Your task:** Complete `create_rag_prompt()` using `PromptTemplate`.

Your template must contain the variables `{context}` and `{question}` and tell the model:
- Answer **only** from the supplied handbook context.
- If the answer is not supported, respond **exactly**: `Insufficient context.`
- Give a short, student-friendly response.
- Do not invent policy details or assume any missing information.

**Expected output:** A LangChain `PromptTemplate` object.

**Purpose of Code:** This function defines the instructions and placeholders for the final RAG question-answering prompt. It tells the LLM to use the retrieved handbook context and avoid inventing answers when the context is insufficient.


In [37]:
def create_rag_prompt() -> PromptTemplate:
    """Create and return a grounded RAG prompt template."""
    # TODO 1: Write a template with {context} and {question}.
    # TODO 2: Specify how to handle missing information.
    # TODO 3: Return a PromptTemplate object.
    template = (
        "Answer the student's question using ONLY the supplied handbook context below. "
        "If the answer is not supported by the context, respond with exact text: Insufficient context.\n"
        "Keep your response short, student-friendly, and do not invent policy details.\n\n"
        "Context:\n{context}\n\n"
        "Question: {question}\n"
        "Answer:"
    )
    return PromptTemplate(input_variables=["context", "question"], template=template)

### Step 10. Convert a LangChain prompt into a DeepSeek API response

**Your task:** Complete `deepseek_model_step(prompt_value)`.

- LangChain passes a prompt value to your model step.
- Convert it into text using `.to_string()`.
- Pass that text into `call_deepseek` as a user message, together with a concise system message.
- Return the generated text.

**Hint:** In the next step you will wrap this Python function using `RunnableLambda`.

**Expected output:** A string. Do not create a new API key or a second client.

**Purpose of Code:** This function acts as the link between LangChain and the DeepSeek API. It converts a LangChain-formatted prompt into an API request and returns the model’s answer to the chain.


In [38]:
def deepseek_model_step(prompt_value: Any) -> str:
    """Convert a LangChain prompt value into a DeepSeek text response."""
    # TODO 1: Convert prompt_value to a plain string.
    prompt_text = prompt_value.to_string()
    # TODO 2: Build system and user messages.
    messages = [
        {"role": "system", "content": "You are a helpful student helpdesk assistant."},
        {"role": "user", "content": prompt_text},
    ]
    # TODO 3: Call call_deepseek and return the response.
    return call_deepseek(messages)

### Step 11. Combine the LangChain components

**Your task:** Complete `build_rag_chain(prompt)`.

Connect three components in this order:

```text
PromptTemplate -> RunnableLambda(deepseek_model_step) -> StrOutputParser
```

- Wrap `deepseek_model_step` in a `RunnableLambda`.
- Create a `StrOutputParser`.
- Use the LangChain pipe operator (`|`) to connect the components.
- Return the assembled chain.

**Expected output:** An object that supports `.invoke({'context': '...', 'question': '...'})`.

**Purpose of Code:** This function connects prompt preparation, the DeepSeek call, and output parsing into one reusable LangChain pipeline. The completed chain can answer different questions by receiving new context and question values.


In [39]:
def build_rag_chain(prompt: PromptTemplate):
    """Return PromptTemplate -> model runnable -> text parser chain."""
    # TODO 1: Create RunnableLambda(deepseek_model_step).
    model_runnable = RunnableLambda(deepseek_model_step)
    # TODO 2: Create StrOutputParser().
    parser = StrOutputParser()
    # TODO 3: Connect all three components with | and return the chain.
    return prompt | model_runnable | parser
    pass

## Part F — Connect and Test the Assistant
### Step 12. Answer one student question

**Your task:** Complete `answer_student_question(question, chunks, chain)`.

- Call `retrieve_best_chunk`.
- If the best score is `0`, return `Insufficient context.` **without** an LLM request.
- Otherwise call `chain.invoke` with `{'context': context, 'question': question}`.
- Return this dictionary:

```python
{
    'question': '...',
    'answer': '...',
    'retrieval_score': 0,
    'retrieved_context': '...'
}
```

**Important:** Even with a positive retrieval score, the model must say `Insufficient context.` when the retrieved chunk does not actually answer the question. Do not treat keyword matches as proof.

**Purpose of Code:** This function combines retrieval and answer generation into the student helpdesk workflow. It searches the handbook, skips the LLM call when no chunk matches, and returns an answer with information about the retrieved context.


In [40]:
def answer_student_question(question: str, chunks: list[str], chain) -> dict:
    """Retrieve handbook context, run the chain, and return a result dict."""
    # TODO 1: Retrieve the best context and overlap score.
    context, score = retrieve_best_chunk(question, chunks)
    # TODO 2: Return insufficient context if retrieval failed.
    if score == 0:
        return {
            "question": question,
            "answer": "Insufficient context.",
            "retrieval_score": score,
            "retrieved_context": context,
        }
    # TODO 3: Otherwise invoke the chain using context and question.
    answer = chain.invoke({"context": context, "question": question})

    # TODO 4: Return the required result dictionary.
    return {
        "question": question,
        "answer": answer,
        "retrieval_score": score,
        "retrieved_context": context,
    }

### Step 13. Run a functional test after implementing your functions

**Your task:** Execute this provided test harness. Fix any failures in your functions before continuing.

It checks extraction and chunking, compares the two classification prompts, builds the chain, and asks two representative policy questions.

**Note:** Run only after completing Steps 2–12. The code below is intentionally **not** a solution to any TODO function.

**Purpose of Code:** This test cell checks whether the functions you completed work together as one assistant. It verifies basic PDF processing, compares classification prompts, and runs sample questions through the RAG chain.


In [41]:
# TEST HARNESS — Run once all TODO functions are complete.
handbook_text = read_handbook_pdf(PDF_PATH)
assert isinstance(handbook_text, str) and handbook_text.strip(), "PDF text must not be empty."

handbook_chunks = split_text_into_chunks(handbook_text)
assert isinstance(handbook_chunks, list) and handbook_chunks, "Chunks must be a non-empty list."

print("Extracted characters:", len(handbook_text))
print("Number of chunks:", len(handbook_chunks))

comparison = compare_classification_prompts("Can I submit my homework late?")
print("\nPrompt comparison:", comparison)

rag_prompt = create_rag_prompt()
rag_chain = build_rag_chain(rag_prompt)

for sample_question in [
    "What is the minimum live attendance required for a certificate?",
    "Does the academy provide students with free laptops?",
]:
    result = answer_student_question(sample_question, handbook_chunks, rag_chain)
    print("\nQuestion:", sample_question)
    print("Answer:", result["answer"])
    print("Retrieval score:", result["retrieval_score"])

Extracted characters: 3086
Number of chunks: 4

Prompt comparison: {'question': 'Can I submit my homework late?', 'zero_shot': 'Assignments', 'few_shot': 'Assignments'}

Question: What is the minimum live attendance required for a certificate?
Answer: The certificate requires at least 80 % live attendance.
Retrieval score: 7

Question: Does the academy provide students with free laptops?
Answer: Insufficient context.
Retrieval score: 3


### Step 14. Evaluate the assistant using a fixed test set

Use the provided six questions. For each response:
- Verify its claims directly against the handbook PDF.
- Label it `Supported`, `Unsupported`, or `Insufficient context` based on the **content**, not just the model's confidence.
- Write a one-sentence observation.
- Highlight at least one limitation of the simple word-overlap retriever.

**Your task:** Complete `evaluate_questions(test_questions, chunks, chain)` to collect the question and result for every test case. You can print your observations or write a Markdown results table below.

**Purpose of Code:** This function runs the assistant on a fixed set of student questions and collects its results. You will use those results to check whether answers are supported by the handbook and identify retrieval limitations.


In [43]:
TEST_QUESTIONS = [
    "What is the deadline for each weekly assignment?",
    "Can a student resubmit a graded assignment?",
    "What attendance is needed for a completion certificate?",
    "How many mentoring sessions can a learner book each month?",
    "Does the program guarantee a job?",
    "Does the academy give every student a free laptop?",
]

def evaluate_questions(test_questions: list[str], chunks: list[str], chain) -> list[dict]:
    """Return a list of assistant result dictionaries for the test questions."""
    results = []
    # TODO 1: Loop through the test questions.
    for question in test_questions:
    # TODO 2: Call answer_student_question for each one.
      result = answer_student_question(question = question, chunks = chunks, chain = chain)
      results.append(result)
    # TODO 3: Append each result to a list and return it.
    return results

eval_results = evaluate_questions(TEST_QUESTIONS, handbook_chunks, rag_chain)

for i, res in enumerate(eval_results, 1):
    print(f"--- Question {i} ---")
    print("Question:", res["question"])
    print("Answer:", res["answer"])
    print("Retrieval score:", res["retrieval_score"])
    print("Retrieved context snippet:", res["retrieved_context"][:150].replace("\n", " "), "...\n")

--- Question 1 ---
Question: What is the deadline for each weekly assignment?
Answer: Each weekly assignment is due by Sunday at 11:59 PM IST.
Retrieval score: 7
Retrieved context snippet: ilable for revision but do not count as live attendance. 4. Weekly Assignment Deadlines Assignments for each teaching week are due by Sunday at 11:59  ...

--- Question 2 ---
Question: Can a student resubmit a graded assignment?
Answer: Yes. A learner can resubmit each graded assignment **once**, as long as it’s done within seven calendar days after receiving feedback.
Retrieval score: 4
Retrieved context snippet: ilable for revision but do not count as live attendance. 4. Weekly Assignment Deadlines Assignments for each teaching week are due by Sunday at 11:59  ...

--- Question 3 ---
Question: What attendance is needed for a completion certificate?
Answer: You need to attend at least 80 % of the live sessions to qualify for the completion certificate.
Retrieval score: 6
Retrieved context snippet: i

### Step 15. Record and explain your findings (Markdown answer)

**Purpose of this step:** This final step does not require new code. It documents what your completed assistant produced, checks its answers against the handbook, and explains the strengths and limitations you observed.

Fill in the table below after running your evaluation function. This final step tests your ability to **interpret results**, not just run code.

| *Test question (short name)* | *Answer supported by handbook?* | *Your observation* |
| --- | --- | --- |
| Weekly deadline | Supported | Correctly retrieved the deadline chunk (score 7) and accurately stated Sunday at 11:59 PM IST. |
| Resubmission | Supported | Correctly retrieved the policy chunk (score 4) and confirmed one resubmission within seven calendar days of feedback. |
| Certificate attendance | Supported | Correctly identified the 80% live session attendance requirement from the retrieved chunk (score 6). |
| Mentoring | Supported | Retrieved the mentoring section (score 7) and accurately reported the limit of two individual sessions per month. |
| Job guarantee | Supported | Retrieved the career/placement policy chunk (score 6) and accurately confirmed that employment or placement is not guaranteed. |
| Free laptop | Insufficient context | Matched common words like "student" and "academy" (score 4), but the model properly returned `Insufficient context.` because free laptops are not in the handbook. |

**Answer these questions:**
1. Did zero-shot and few-shot prompting produce the same category? Explain briefly.<br>
**ANS:**
Zero-shot and few-shot prompting can produce the same category when the student's question is clear. However, few-shot prompting provides examples that help the model understand the expected classification pattern. This can improve consistency and accuracy, especially for ambiguous questions.
2. Why does the assistant retrieve a chunk before calling the LLM?<br>
**ANS:**
The assistant retrieves a relevant chunk from the student handbook before calling the LLM because it needs accurate information to generate its response. This approach is called Retrieval-Augmented Generation (RAG). It helps the model answer questions using the provided document instead of relying only on its general knowledge, reducing the chances of generating incorrect or unsupported information.
3. Give one example of a question that keyword-overlap retrieval might handle poorly.<br>
**ANS:**
A student might ask, "What do I need to do to receive my course completion document?" while the handbook uses the term "certificate eligibility."
Although both questions have similar meanings, they use different words. A simple keyword-overlap retrieval system might fail to identify the correct chunk because it matches shared words rather than understanding the actual meaning of the question.
4. Did your assistant invent any policy? If so, how did you improve the prompt or test?<br>
**ANS:**
The assistant may generate unsupported information when the retrieved context does not contain sufficient details to answer a question. To reduce this problem, I instructed the model to answer only using the supplied handbook context and return exactly "Insufficient context." when the information is unavailable.

## Submission Checklist


Upload your completed `.ipynb` with executed outputs on GitHub and share the link of repository.



- [x] All 12 unfinished Python functions have been implemented (Steps 2–12 and Step 14).
- [x] The notebook runs sequentially from a fresh Colab runtime after adding the API key and uploading the PDF.
- [x] The zero-shot/few-shot comparison has an output.
- [x] The PDF has been extracted and chunked successfully.
- [x] A LangChain pipeline produces a grounded answer.
- [x] The assistant handles a question that is not answered in the PDF.
- [x] All six evaluation questions and the reflection are completed.
- [x] The DeepSeek API key is not embedded in code or visible in notebook output.

### Assessment Rubric

| Criterion | Marks |
|---|---:|
| Zero-shot and few-shot prompt quality | 20 |
| Direct LLM API integration | 20 |
| LangChain chain construction | 20 |
| PDF processing, retrieval, grounded answers | 25 |
| Testing, observations, and submission quality | 15 |
| **Total** | **100** |

### Helpful references
- DeepSeek API: https://api-docs.deepseek.com/
- LangChain: https://python.langchain.com/
- PyPDF: https://pypdf.readthedocs.io/
